# PySpark drills 10: batch pipelines

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-drills-10-batch-pipelines.ipynb)

Practice for [PySpark 10: batch pipelines](pyspark-10-batch-pipelines.ipynb). Each exercise uses the `orders` table or a small input from its data cell.

Write your answer in the empty cell under each exercise, run it, and compare your output with the expected output.
The reference answer is hidden under the exercise; open it when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `orders` (one row per order; order 12 lands just after midnight UTC): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#setup-and-sample-data).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F
from pyspark.sql import Window

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

## Exercise 1: Keep the latest change per key

**Difficulty:** Medium · about 10 min

A change feed has several versions per order and one replayed change. Keep the change with the highest `lsn` per
order, breaking ties by the latest `ingested_at`. Return `order_id`, `status` and `lsn`, ordered by `order_id`.

**Sample input**

**`changes`**

| order_id | status | lsn | ingested_at |
| --- | --- | --- | --- |
| 1 | placed | 1 | 2026-01-05 09:00:00 |
| 1 | paid | 2 | 2026-01-05 09:05:00 |
| 2 | placed | 1 | 2026-01-05 10:00:00 |
| 2 | cancelled | 2 | 2026-01-05 10:01:00 |
| 2 | cancelled | 2 | 2026-01-05 10:09:00 |

**Expected output**

| order_id | status | lsn |
| --- | --- | --- |
| 1 | paid | 2 |
| 2 | cancelled | 2 |

In [ ]:
# Sample data for this exercise.
changes = spark.createDataFrame(
    [(1, "placed", 1, datetime(2026, 1, 5, 9, 0)), (1, "paid", 2, datetime(2026, 1, 5, 9, 5)),
     (2, "placed", 1, datetime(2026, 1, 5, 10, 0)), (2, "cancelled", 2, datetime(2026, 1, 5, 10, 1)),
     (2, "cancelled", 2, datetime(2026, 1, 5, 10, 9))],
    "order_id int, status string, lsn int, ingested_at timestamp",
)

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
newest = Window.partitionBy("order_id").orderBy(F.desc("lsn"), F.desc("ingested_at"))
result = (
    changes.withColumn("rn", F.row_number().over(newest)).where("rn = 1")
    .select("order_id", "status", "lsn").orderBy("order_id")
)
result.show()

## Exercise 2: An idempotent upsert

**Difficulty:** Medium · about 15 min

Write `upsert(target, changes)` that replaces target rows whose `order_id` appears in `changes` and adds new ones.
Apply `batch` to `target` twice and return the result ordered by `order_id`.

**Sample input**

**`target`**

| order_id | status |
| --- | --- |
| 1 | paid |
| 2 | placed |
| 3 | shipped |

**`batch`**

| order_id | status |
| --- | --- |
| 2 | cancelled |
| 4 | placed |

**Expected output**

| order_id | status |
| --- | --- |
| 1 | paid |
| 2 | cancelled |
| 3 | shipped |
| 4 | placed |

In [ ]:
# Sample data for this exercise.
target = spark.createDataFrame([(1, "paid"), (2, "placed"), (3, "shipped")], "order_id int, status string")
batch = spark.createDataFrame([(2, "cancelled"), (4, "placed")], "order_id int, status string")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
def upsert(target, changes, key="order_id"):
    return target.join(changes.select(key), key, "left_anti").unionByName(changes)


result = upsert(upsert(target, batch), batch).orderBy("order_id")
result.show()

## Exercise 3: Apply an SCD type 2 change

**Difficulty:** Hard · about 25 min

`dim` holds one current row per customer. Apply `changes`: when a customer's region differs from the current row,
close that row (`valid_to` = `changed_on`, `is_current` = false) and add a new current row; new customers get a
current row; unchanged customers are left alone. Return all rows ordered by `customer_id` then `valid_from`.

**Sample input**

**`dim`**

| customer_id | region | valid_from | valid_to | is_current |
| --- | --- | --- | --- | --- |
| 1 | North | 2024-01-03 | null | true |
| 2 | North | 2024-02-11 | null | true |

**`changes`**

| customer_id | region | changed_on |
| --- | --- | --- |
| 1 | West | 2026-01-15 |
| 2 | North | 2026-01-15 |
| 3 | South | 2026-01-15 |

**Expected output**

| customer_id | region | valid_from | valid_to | is_current |
| --- | --- | --- | --- | --- |
| 1 | North | 2024-01-03 | 2026-01-15 | false |
| 1 | West | 2026-01-15 | null | true |
| 2 | North | 2024-02-11 | null | true |
| 3 | South | 2026-01-15 | null | true |

In [ ]:
# Sample data for this exercise.
dim = spark.createDataFrame(
    [(1, "North", date(2024, 1, 3), None, True), (2, "North", date(2024, 2, 11), None, True)],
    "customer_id int, region string, valid_from date, valid_to date, is_current boolean",
)
changes = spark.createDataFrame(
    [(1, "West", date(2026, 1, 15)), (2, "North", date(2026, 1, 15)), (3, "South", date(2026, 1, 15))],
    "customer_id int, region string, changed_on date",
)

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
current = dim.where("is_current").select("customer_id", F.col("region").alias("current_region"))
changed = changes.join(current, "customer_id", "left").where(~F.col("region").eqNullSafe(F.col("current_region")))
closed = (
    dim.join(changed.select("customer_id", "changed_on"), "customer_id", "left")
    .withColumn("closes", F.col("is_current") & F.col("changed_on").isNotNull())
    .withColumn("valid_to", F.when(F.col("closes"), F.col("changed_on")).otherwise(F.col("valid_to")))
    .withColumn("is_current", F.col("is_current") & ~F.col("closes"))
    .drop("changed_on", "closes")
)
opened = changed.select("customer_id", "region", F.col("changed_on").alias("valid_from"),
                        F.lit(None).cast("date").alias("valid_to"), F.lit(True).alias("is_current"))
result = closed.unionByName(opened).orderBy("customer_id", "valid_from")
result.show()

## Exercise 4: Quarantine bad rows with a reason

**Difficulty:** Medium · about 15 min

Check `incoming` against three rules, in this order: `order_id` is not null, `amount` is not null, `amount` is not
negative. Return `valid` (`order_id`, `amount` of passing rows) and `quarantine` (`order_id`, `amount`, `reason`),
each ordered by `amount`.

**Sample input**

**`incoming`**

| order_id | amount |
| --- | --- |
| 1 | 50.00 |
| null | 5.00 |
| 3 | null |
| 4 | -3.00 |
| 5 | 8.00 |

**Expected output**

| order_id | amount |
| --- | --- |
| 5 | 8.00 |
| 1 | 50.00 |

| order_id | amount | reason |
| --- | --- | --- |
| 3 | null | missing amount |
| 4 | -3.00 | negative amount |
| null | 5.00 | missing order_id |

In [ ]:
# Sample data for this exercise.
incoming = spark.createDataFrame(
    [(1, D("50.00")), (None, D("5.00")), (3, None), (4, D("-3.00")), (5, D("8.00"))], "order_id int, amount decimal(12,2)"
)

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
checked = incoming.withColumn(
    "reason",
    F.when(F.col("order_id").isNull(), "missing order_id")
    .when(F.col("amount").isNull(), "missing amount")
    .when(F.col("amount") < 0, "negative amount"),
)
valid = checked.where("reason IS NULL").select("order_id", "amount").orderBy("amount")
quarantine = checked.where("reason IS NOT NULL").select("order_id", "amount", "reason").orderBy(F.col("amount").asc_nulls_first())
valid.show()
quarantine.show()

## Exercise 5: Rewrite one day only

**Difficulty:** Medium · about 15 min

`daily` is written to `path` partitioned by `order_date`. Replace only 12 January with `fixed`, leaving the other days
untouched, and return the table's `order_date` and `revenue` ordered by date.

**Expected output**

| order_date | revenue |
| --- | --- |
| 2026-01-05 | 234.50 |
| 2026-01-06 | 260.00 |
| 2026-01-07 | 204.25 |
| 2026-01-09 | 310.00 |
| 2026-01-10 | 89.90 |
| 2026-01-12 | 1000.00 |
| 2026-01-13 | 210.00 |
| 2026-01-14 | 96.40 |

In [ ]:
# Sample data for this exercise.
path = os.path.join(WORK_DIR, "drill_daily")
daily = orders.groupBy(F.to_date("order_ts").alias("order_date")).agg(F.sum("amount").alias("revenue"))
daily.write.mode("overwrite").partitionBy("order_date").parquet(path)
fixed = spark.createDataFrame([(date(2026, 1, 12), D("1000.00"))], "order_date date, revenue decimal(22,2)")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
try:
    fixed.write.mode("overwrite").partitionBy("order_date").parquet(path)
finally:
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")
result = spark.read.parquet(path).select("order_date", "revenue").orderBy("order_date")
result.show()

**Next:** [PySpark 11: structured streaming](pyspark-11-structured-streaming.ipynb)